# Camada Silver - US Holiday Dates (Diogo)

Limpa a tabela Bronze dos feriados e grava a tabela Silver em Delta, na base `silver` do grupo.

| Tabela Silver | 1 linha = | Usada em |
|---|---|---|
| `silver.feriados_eua` | 1 feriado dos EUA entre junho e dezembro de 2017 | `dim_calendario` (Gold); Q3 do Diogo (atrasos em feriados), KPI 3 do Henrique e Q2 do Quévin |

Tratamentos (mesma lógica do notebook `Feriados_Silver` do Diogo):
- `Date` passa a `DATE` (`data`); ficam só os feriados do intervalo do grupo (2017-06-01 a 2017-12-31);
- `WeekDay`, `Month`, `Day` e `Year` são eliminados, porque derivam da data (em 6 linhas "Labor Day Weekend" o mês e o dia nem coincidem com a data);
- no nome do feriado (`feriado`) tiram-se os espaços a mais e uniformiza-se o apóstrofo ("New Year’s Eve" → "New Year's Eve");
- duplicados (mesma data e mesmo feriado) são removidos.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - US Holiday Dates (Diogo)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Intervalo de tempo comum do grupo (README): só entram registos de 1 de junho a 31 de dezembro de 2017
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

In [4]:
fer = spark.table("bronze.us_holidays")
print("Feriados na Bronze:", fer.count())

# month/day com erros em algumas linhas: são eliminados (derivam da data)
erradas = fer.filter((F.month(F.to_date("Date")) != F.col("Month").cast("int")) |
                     (F.dayofmonth(F.to_date("Date")) != F.col("Day").cast("int"))).count()
print("Linhas em que Month/Day não batem com a data:", erradas)

feriados = (fer
    .select(F.to_date(F.trim(F.col("Date")), "yyyy-MM-dd").alias("data"),
            F.regexp_replace(F.regexp_replace(F.trim(F.col("Holiday")), "’", "'"), r"\s+", " ").alias("feriado"))
    .dropDuplicates(["data", "feriado"]))
assert feriados.filter(F.col("data").isNull() | F.col("feriado").isNull()).count() == 0

antes = feriados.count()
feriados = feriados.filter(F.col("data").between(DATA_INICIO, DATA_FIM)).orderBy("data")
print(f"Removidos {antes - feriados.count()} feriados fora de {DATA_INICIO} a {DATA_FIM}. Ficam {feriados.count()}.")
feriados.show(truncate=False)
gravar_silver(feriados, "feriados_eua")

Feriados na Bronze: 342
Linhas em que Month/Day não batem com a data: 6
Removidos 330 feriados fora de 2017-06-01 a 2017-12-31. Ficam 12.
+----------+-----------------+
|data      |feriado          |
+----------+-----------------+
|2017-06-19|Juneteenth       |
|2017-07-04|4th of July      |
|2017-09-02|Labor Day Weekend|
|2017-09-03|Labor Day Weekend|
|2017-09-04|Labor Day        |
|2017-10-09|Columbus Day     |
|2017-11-11|Veterans Day     |
|2017-11-22|Thanksgiving Eve |
|2017-11-23|Thanksgiving Day |
|2017-12-24|Christmas Eve    |
|2017-12-25|Christmas Day    |
|2017-12-31|New Year's Eve   |
+----------+-----------------+

silver.feriados_eua: 12 linhas


In [5]:
spark.stop()